# 06 - Gold Layer - Date Dimension

Create a business-ready Date Dimension from the validated Silver calendar data.

**Source:** `end-to-end_pipeline.silver.calendar`
**Target:** `end-to-end_pipeline.gold.dim_date`

**Model Role:** Dimension Table
**Business Key:** `date`
**Approach:** Profile → Inspect → Transform → Validate

**Purpose:**
Provide standardized date attributes for time-based analysis such as year, quarter, month, week, weekday, and weekend analysis.

## Cell 1 - Profile Silver Calendar Data

**Description:**
Confirm that the Silver calendar table is complete and suitable for the Gold Date Dimension. This checks date uniqueness, row count, range, and availability of the main time attributes.


In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER CALENDAR FOR GOLD MODELING
-- Purpose: Confirm date grain, uniqueness,
--          completeness, and date range
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date) AS distinct_dates,
    COUNT(*) - COUNT(DISTINCT date) AS duplicate_dates,

    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END)
        AS null_dates,

    COUNT(DISTINCT year)
        AS years,

    COUNT(DISTINCT quarter)
        AS quarters,

    COUNT(DISTINCT month_number)
        AS months,

    COUNT(DISTINCT day_name)
        AS day_names,

    MIN(date) AS earliest_date,
    MAX(date) AS latest_date

FROM `end-to-end_pipeline`.silver.calendar;

total_rows,distinct_dates,duplicate_dates,null_dates,years,quarters,months,day_names,earliest_date,latest_date
1826,1826,0,0,5,4,12,7,2021-01-01,2025-12-31


## Cell 2 - Inspect Date Business Attributes

**Description:**
Review the distribution of dates by year and quarter. This confirms that the Date Dimension supports historical and year-over-year analysis across the full 2021–2025 period.

In [0]:
%sql

-- ============================================================
-- CELL 2: INSPECT DATE BUSINESS ATTRIBUTES
-- Purpose: Review date distribution by year and quarter
-- ============================================================

SELECT
    year,
    quarter,
    COUNT(*) AS number_of_days

FROM `end-to-end_pipeline`.silver.calendar

GROUP BY
    year,
    quarter

ORDER BY
    year,
    quarter;

year,quarter,number_of_days
2021,Q1,90
2021,Q2,91
2021,Q3,92
2021,Q4,92
2022,Q1,90
2022,Q2,91
2022,Q3,92
2022,Q4,92
2023,Q1,90
2023,Q2,91


## Cell 3 - Transform Silver → Gold Date Dimension

**Description:**
Create the Gold Date Dimension at **one row per calendar date**.

Silver cleaning is not repeated. This step exposes the time attributes needed for business reporting and keeps `date` as the key that will later connect to `gold.fact_sales`.


In [0]:
%sql

-- ============================================================
-- CELL 3: CREATE GOLD DATE DIMENSION
-- Grain: One row per calendar date
-- Business Key: date
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_date AS

SELECT
    date,
    year,
    quarter,
    month_number,
    month_name,
    week_number,
    day_name,
    is_weekend

FROM `end-to-end_pipeline`.silver.calendar;

num_affected_rows,num_inserted_rows


## Cell 3a - Add Business Metadata

**Description:**
Add table and column comments to improve discoverability in Genie and Databricks dashboards. These descriptions help users understand the business meaning of each time attribute.

In [0]:
%sql

-- ============================================================
-- CELL 3a: ADD BUSINESS METADATA TO DATE DIMENSION
-- Purpose: Improve discoverability for Genie and dashboards
-- ============================================================

COMMENT ON TABLE `end-to-end_pipeline`.gold.dim_date IS 
'Date dimension providing calendar attributes for time-based analysis and reporting. One row per calendar date.';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN date COMMENT 'Calendar date (business key for time-based joins)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN year COMMENT 'Calendar year (YYYY)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN quarter COMMENT 'Calendar quarter (1-4)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN month_number COMMENT 'Month number within year (1-12)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN month_name COMMENT 'Full month name (e.g., January, February)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN week_number COMMENT 'ISO week number within year (1-53)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN day_name COMMENT 'Full day name (e.g., Monday, Tuesday)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_date 
  ALTER COLUMN is_weekend COMMENT 'Weekend indicator (TRUE for Saturday/Sunday, FALSE otherwise)';

## Cell 4 - Validate Gold Date Dimension

**Description:**
Validate that the Gold Date Dimension contains one unique row per date, covers the complete 2021–2025 period, and contains no gaps in the calendar.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD DATE DIMENSION
-- Purpose: Confirm date uniqueness, completeness,
--          continuity, and Silver → Gold consistency
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT date)
            AS distinct_dates,

        COUNT(*) - COUNT(DISTINCT date)
            AS duplicate_dates,

        SUM(
            CASE
                WHEN date IS NULL THEN 1
                ELSE 0
            END
        ) AS null_dates,

        COUNT(DISTINCT year)
            AS year_count,

        COUNT(DISTINCT quarter)
            AS quarter_count,

        COUNT(DISTINCT month_number)
            AS month_count,

        COUNT(DISTINCT day_name)
            AS day_name_count,

        MIN(date) AS earliest_date,
        MAX(date) AS latest_date,

        DATEDIFF(MAX(date), MIN(date)) + 1 - COUNT(*)
            AS missing_dates

    FROM `end-to-end_pipeline`.gold.dim_date
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.calendar
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.total_rows = 1826
            AND v.distinct_dates = 1826
            AND v.duplicate_dates = 0
            AND v.null_dates = 0
            AND v.year_count = 5
            AND v.quarter_count = 4
            AND v.month_count = 12
            AND v.day_name_count = 7
            AND v.missing_dates = 0
            AND v.earliest_date = DATE '2021-01-01'
            AND v.latest_date = DATE '2025-12-31'
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;

total_rows,distinct_dates,duplicate_dates,null_dates,year_count,quarter_count,month_count,day_name_count,earliest_date,latest_date,missing_dates,silver_rows,validation_status
1826,1826,0,0,5,4,12,7,2021-01-01,2025-12-31,0,1826,PASS
